# BioAgents++ Experiments

This notebook demonstrates the execution of the BioAgents++ pipeline, including the Verification Agent and Confidence Estimator, on a set of synthetic workflows.

In [1]:
import sys
import os
import json

# Add parent directory to path to import modules
sys.path.append(os.path.abspath('..'))

from agents.planner import PlannerAgent
from agents.selector import ToolSelectorAgent
from agents.generator import WorkflowGeneratorAgent
from agents.verifier import VerificationAgent
from agents.confidence import ConfidenceEstimator
from evaluation.metrics import MetricsEvaluator

In [2]:
# Load synthetic workflows
with open('../data/synthetic_workflows.json', 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} workflows.")

Loaded 6 workflows.


In [3]:
# Initialize Agents
planner = PlannerAgent()
selector = ToolSelectorAgent()
generator = WorkflowGeneratorAgent()
verifier = VerificationAgent()
confidence = ConfidenceEstimator()
evaluator = MetricsEvaluator()

In [4]:
# Run Pipeline on all workflows
results = []
for item in data:
    query = item['query']
    gold = item['gold_workflow']
    
    print(f"\nProcessing Query: '{query}'")
    
    # Agent 1: Planner
    task_info = planner.plan(query)
    
    # Agent 2: Selector
    tools_info = selector.select_tools(task_info)
    
    # Agent 3: Generator
    workflow_info = generator.generate(tools_info)
    
    # NEW Agent 4: Verifier
    verification_info = verifier.verify(workflow_info, task_info)
    
    # NEW Agent 5: Confidence Estimator
    conf_info = confidence.estimate(verification_info, workflow_info)
    
    # Evaluation
    eval_metrics = evaluator.evaluate(workflow_info['workflow'], gold)
    
    print(f"Generated Workflow: {workflow_info['workflow']}")
    print(f"Verification: {verification_info}")
    print(f"Confidence: {conf_info['confidence']}")
    print(f"Metrics: {eval_metrics}")
    print("-" * 50)



Processing Query: 'How do I identify differentially expressed genes from my RNA-seq FASTQ files?'
Generated Workflow: ['FastQC', 'STAR', 'DESeq2']
Verification: {'valid': False, 'missing': ['featureCounts'], 'invalid': []}
Confidence: 0.8
Metrics: {'accuracy_f1': 0.86, 'completeness': 0.75, 'hallucination_rate': 0.0}
--------------------------------------------------

Processing Query: 'Please provide a standard RNA-seq pipeline.'
Generated Workflow: ['FastQC', 'STAR', 'DESeq2']
Verification: {'valid': False, 'missing': ['featureCounts'], 'invalid': []}
Confidence: 0.8
Metrics: {'accuracy_f1': 0.86, 'completeness': 0.75, 'hallucination_rate': 0.0}
--------------------------------------------------

Processing Query: 'I have Whole Genome Sequencing data, how to get VCF?'
Generated Workflow: ['FastQC', 'BWA', 'GATK', 'VCF Output']
Verification: {'valid': False, 'missing': ['Samtools'], 'invalid': []}
Confidence: 0.8
Metrics: {'accuracy_f1': 0.89, 'completeness': 0.8, 'hallucination_rate